<a href="https://colab.research.google.com/github/mrm6676/March-ML-mania-26/blob/main/F1_PitStop_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


1.   Import libraries

In [ ]:
# Install catboost if not already installed
!pip install catboost

import numpy as np
import pandas as pd
import xgboost as XGB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.simplefilter('ignore')
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from catboost import CatBoostClassifier


2.   Load data

In [ ]:
# Load datasets
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')
submission_df = pd.read_csv('/content/sample_submission.csv')

print(f"Train Shape: {train.shape}")
print(f"Test Shape: {test.shape}")

In [ ]:
print(train.columns.tolist())
train.head(10)

In [ ]:
def process(df):
    numeric = df.select_dtypes(include=[np.number]).columns
    df[numeric] = df[numeric].fillna(df[numeric].median())

    categorical = df.select_dtypes(include=['object']).columns
    df[categorical] = df[categorical].fillna('None')

    # Apply OrdinalEncoder to categorical columns
    for col in categorical:
        encoder = OrdinalEncoder()
        df[col] = encoder.fit_transform(df[[col]])

    return df

train = process(train)
test = process(test)




3.   Data Visualization




In [ ]:
#Correlation Heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(train.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

4.   Feature Engineering

In [ ]:
def process_and_engineer(train_df, test_df):
    target = 'PitNextLap'
    features = [feat for feat in test_df.columns if feat != 'id']
    cat_cols = test_df.select_dtypes(include=['object', 'category']).columns.tolist()

    X = train_df[features].copy()
    y = train_df[target].copy()
    X_test = test_df[features].copy()

    for col in cat_cols:
        freq_map = X[col].value_counts(normalize=True).to_dict()
        X[f'{col}_freq'] = X[col].map(freq_map)
        X_test[f'{col}_freq'] = X_test[col].map(freq_map).fillna(0)

    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X[cat_cols] = encoder.fit_transform(X[cat_cols].astype(str)).astype(int)
    X_test[cat_cols] = encoder.transform(X_test[cat_cols].astype(str)).astype(int)

    X['Life_Ratio'] = X['TyreLife'] / (X['LapNumber'] + 1)
    X_test['Life_Ratio'] = X_test['TyreLife'] / (X_test['LapNumber'] + 1)

    return X, y, X_test

X, y, X_test = process_and_engineer(train, test)

5.   Train baseline Models

In [ ]:
folds = StratifiedKFold(n_splits = 5, shuffle=True, random_state=42)
oof_lgb, oof_xgb, oof_cat = [np.zeros(len(X)) for _ in range(3)]
preds_lgb, preds_xgb, preds_cat = [np.zeros(len(X_test)) for _ in range(3)]

xgb_params = {
    'n_estimators': 5000,
    'learning_rate': 0.0468,
    'max_depth': 9,
    'min_child_weight': 2,
    'subsample': 0.577,
    'colsample_bytree': 0.641,
    'gamma': 0.720,
    'tree_method': 'hist',
    'device': 'cuda' if XGB.__version__ >= '2.0' else 'gpu',
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'early_stopping_rounds': 300
}

lgb_params = {
    'n_estimators': 5000,
    'learning_rate': 0.01,
    'num_leaves': 255,
    'max_depth': -1,
    'metric': 'auc',
     'verbosity': -1,
    'early_stopping_round': 300
}

cat_params = {
    'iterations': 5000,
    'learning_rate': 0.01,
    'depth': 8,
    'l2_leaf_reg': 3,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': 42,
    'verbose': 0,
    'early_stopping_rounds': 300,
    'task_type': 'CPU'
}

for fold, (t_idx, v_idx) in enumerate(folds.split(X, y)):
    xt, xv = X.iloc[t_idx], X.iloc[v_idx]
    yt, yv = y.iloc[t_idx], y.iloc[v_idx]

    # XGBoost
    m_xgb = XGB.XGBClassifier(**xgb_params)
    m_xgb.fit(xt, yt, eval_set=[(xv, yv)], verbose=False)
    oof_xgb[v_idx] = m_xgb.predict_proba(xv)[:, 1]
    preds_xgb += m_xgb.predict_proba(X_test)[:, 1] / folds.n_splits

    # LightGBM
    m_lgb = lgb.LGBMClassifier(**lgb_params)
    m_lgb.fit(xt, yt, eval_set=[(xv, yv)], callbacks=[lgb.early_stopping(300)])
    oof_lgb[v_idx] = m_lgb.predict_proba(xv)[:, 1]
    preds_lgb += m_lgb.predict_proba(X_test)[:, 1] / folds.n_splits

    # CatBoost
    m_cat = CatBoostClassifier(**cat_params)
    m_cat.fit(xt, yt, eval_set=(xv, yv), early_stopping_rounds=300, verbose=False)
    oof_cat[v_idx] = m_cat.predict_proba(xv)[:, 1]
    preds_cat += m_cat.predict_proba(X_test)[:, 1] / folds.n_splits

    print(f"Fold {fold+1} Completed")

# Blending
final_oof = (oof_lgb * 0.4) + (oof_xgb * 0.3) + (oof_cat * 0.3)
final_preds = (preds_lgb * 0.4) + (preds_xgb * 0.3) + (preds_cat * 0.3)
print(f"Ensemble AUC: {roc_auc_score(y, final_oof):.5f}")

In [ ]:
#ROC curve graph
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
plot_df = pd.DataFrame({
    'xgb': oof_xgb,
    'lgb': oof_lgb,
    'cat': oof_cat,
    'y': y
})
plt.figure(figsize=(10, 8))
for col in plot_df.columns[:-1]:
    fpr, tpr, _ = roc_curve(plot_df['y'], plot_df[col])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{col}, AUC={roc_auc:.3f}')
    plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc="lower right")
plt.show()

In [ ]:
from scipy.stats import rankdata

final_preds_weighted = (preds_lgb * 0.4) + (preds_xgb * 0.3) + (preds_cat * 0.3)

rank_preds = (
    rankdata(preds_lgb) * 0.4 +
    rankdata(preds_xgb) * 0.3 +
    rankdata(preds_cat) * 0.3
)
final_preds_ranked = rank_preds / (len(rank_preds) + 1)

print(f"Weighted Mean Sample: {final_preds_weighted[:5]}")
print(f"Rank-Averaged Sample: {final_preds_ranked[:5]}")


6.   Final Submission



In [ ]:
submission = pd.DataFrame({
    'id': test['id'],
    'PitNextLap': rankdata(final_preds) / (len(final_preds) + 1)
})

submission.to_csv('submission.csv', index=False)